---
title: Building C A-7.5 \& A-A Transfer Beam
project: Beaverton Creak Apartments
location: Beaverton, OR
client: GBD Architects
job_number: 10022500446
author:
  name: Andy Sazima
  initials: AGS
  email: andy.sazima@kpff.com
---

In [1]:
%reset -f
import math
import os
import warnings
from collections.abc import Callable
from math import pow, sqrt

import forallpeople as units
import handcalcs
import handcalcs.render
import numpy as np

# import polars as pl
import pandas as pd
import sympy as sp
from Pynite import FEModel3D
from structural_tools.notebook.display import display_table, display_text, sig_figs, display_math
from structural_tools.notebook.calculation import check_value, set_params_columns, feet_inches as fi
from structural_tools.notebook.plotting import set_plot_style, plot_internal_diagram
from structural_tools.steel import calculate_critical_stress
from copy import deepcopy
import matplotlib.pyplot as plt

from Pynite import FEModel3D
from steelpy import aisc

In [2]:
units.environment(env_name="structural", top_level=True)
handcalcs.set_option("custom_symbols", {"__": ",", "plus": "+", "minus": "-"})
handcalcs.set_option("greek_exclusions", ["psi"])
handcalcs.set_option("display_precision", 4)
handcalcs.set_option("param_columns", 1)
handcalcs.set_option("latex_block_start", "\\begin{equation*}")
handcalcs.set_option("latex_block_end", "\\end{equation*}")
set_plot_style()

In [ ]:
BUILDING = "c"
GENERATE_NEW_IMAGES = False

# Loading

We have the following loading conditions:

In [4]:
set_params_columns(6)

In [5]:
%%render params
DL = 28.5 * psf
LL = 40 * psf

<IPython.core.display.Latex object>

Since we are designing interior beams, live load reduction according to ASCE 7 Section 4.7 will apply. Each beam will have a different reduction factor, but the following applies:

1. Interior beam $\rightarrow K_{LL} = 2$ (ASCE 7 Table 4.7-1)
2. The transfer beams all support at least 2 floors $\rightarrow L_{red} \geq 0.4 L_o = L_{red,limit}$ (ASCE 7 Sec. 4.7.2)

In [6]:
K_LL = 2
L_red__limit = 0.4 * LL

We will assume that the top flanges are fully braced by the floor (no LTB).

We will first try to design this beam with a single span. If that does not work, we will use 2 spans with a post in the center wall. Dimensions of the problem are shown in \autoref{fig:c_fb_1}.


![Tributary diagram of the beam in question](images/c_fb_1.png){width=50% #fig:c_fb_1}

# One Span

## Beam design

The beam loading and geometry are as follows

In [7]:
%%render
w_trib = fi(26, 6) / 2 * ft
l_trib = fi(7, 9) * ft
n_levels = 1

<IPython.core.display.Latex object>

In [8]:
%%render
A_trib = w_trib * l_trib * n_levels

<IPython.core.display.Latex object>

We can check if the live load can be reduced.

In [9]:
%%render
# fmt: off
if A_trib * K_LL >= 400 * ft * ft: L_red = LL * (0.25 + 15 / sqrt(K_LL * A_trib))
else: L_red = LL


<IPython.core.display.Latex object>

In [10]:
%%render
# fmt: off
if L_red >= L_red__limit: L_red = L_red
else: L_red = L_red__limit

<IPython.core.display.Latex object>

The beam at A-7.5 and A-A resolves the load from levels 2, 3, and 4 (not the roof); therefore, the load combinations are as follows:

Strength

1. $1.4 D$
2. $1.2 D + 1.6 L$

This means the load cases are:

In [11]:
%%render params
D = DL
L = L_red

<IPython.core.display.Latex object>

In [12]:
%%render
D_line = D * w_trib
L_line = L * w_trib

<IPython.core.display.Latex object>

Along with the self weight of the beam, which will be automatically incorporated in the analysis.

In [ ]:
# Initialize FE Model
base_model = FEModel3D()

# Materials
E = 1700  # Modulus of elasticity (ksi)
Fy = 2.400  # Yield stress (ksi)
nu = 0.3  # Poisson's ratio
G = E / (2 * (1 + nu))  # Shear modulus of elasticity (ksi)
rho = 33 / (12**3) / 1000  # Density (kci)
glulam = base_model.add_material(name="glulam", E=E, G=G, nu=nu, rho=rho, fy=Fy)

# Load combinations
base_model.add_load_combo("1.4D", {"D": 1.4}, ["strength"])
base_model.add_load_combo("1.2D+1.6L", {"D": 1.2, "L": 1.6}, ["strength"])

In [16]:
tb_1_glulam = deepcopy(base_model)

# Nodes
tb_1_glulam.add_node("n1", 0, 0, 0)
tb_1_glulam.add_node("n2", float(l_trib.to("inch")), 0, 0)

# Provide simple supports
tb_1_glulam.def_support("n1", True, True, True, False, False, False)
tb_1_glulam.def_support("n2", False, True, True, True, False, False)

# Wide flange shape
display_text(f"Try a 5.5x11-7/8 24F-V4 glulam")
b_glulam = 5.5
d_glulam = 11.875  # minimum that works by trial and error (11.25 is close)

# Add a section with the following properties:
tb_1_glulam.add_section(
    "glulam_5.5_11.875",
    A=b_glulam * d_glulam,
    Iz=1 / 12 * b_glulam * d_glulam**3,
    Iy=1 / 12 * d_glulam * b_glulam**3,
    J=1,
)

# Add a member
tb_1_glulam.add_member("m1", "n1", "n2", "glulam", "glulam_5.5_11.875")

# Uniform loads
total_dead_load = n_levels * float(D_line) / 1000 / 12  # kip/in
tb_1_glulam.add_member_dist_load("m1", "Fy", -total_dead_load, -total_dead_load, case="D")

tb_1_glulam.add_member_self_weight("FY", -1, case="D")

total_live_load = n_levels * float(L_line) / 1000 / 12  # kip/in
tb_1_glulam.add_member_dist_load("m1", "Fy", -total_live_load, -total_live_load, case="L")

# Analyze the beam
tb_1_glulam.analyze()

'Try a 5.5x11-7/8 24F-V4 glulam'

## Post Design

### 6x6 Timber Post

We will now want to check that the 6x6 wood posts are adequate.

The design assumptions we will use are:

1. non-eccentric loading
2. Douglas Fir No. 1
3. Dry

The loading stress on the post, $f_c$, is

In [53]:
max_reaction = max([
    abs(tb_1_glulam.nodes[node].RxnFY[combo])
    for node in tb_1_glulam.nodes.keys()
    for combo in tb_1_glulam.load_combos.keys()
])

In [54]:
%%render
b = 5.5 * inch
d = 5.5 * inch
A = b * d
P = max_reaction * kip
f_c = P / A


<IPython.core.display.Latex object>

In [55]:
set_params_columns(3)

We must adjust the design compressive value, $F_c$

In [56]:
%%render params
C_D = 1.0
C_M = 1.0
C_t = 1.0
C_F = 1.0
C_i = 1.0
C_fu = 1.0
C_T = 1.0

F_c = 1000 * psi
E_min = 580_000 * psi

<IPython.core.display.Latex object>

The column stability factor, $C_P$, is

In [57]:
%%render
F_star_C = F_c * C_D * C_M * C_t * C_F * C_i
E_prime_min = E_min * C_M * C_t * C_fu * C_i * C_T

<IPython.core.display.Latex object>

In [58]:
%%render
# fmt: off
c = 0.8  # sawn lumber
h_level_1 = 11.5 * ft
K_e = 1  # pinned ends
l_e = (K_e * h_level_1.to("inch"))
check_le__d = check_value(l_e / d, 50, "<=")
F_cE = (0.822 * E_prime_min) / (l_e / d) ** 2

<IPython.core.display.Latex object>

In [59]:
C_P = (1 + (F_cE / F_star_C)) / (2 * c) - sqrt(((1 + (F_cE / F_star_C)) / (2 * c)) ** 2 - (F_cE / F_star_C) / c)

The fully adjusted compressive design value is

In [60]:
%%render
F_prime_C = F_star_C * C_P

<IPython.core.display.Latex object>

In [63]:
%%render
check_compression = check_value(f_c.to("psi"), F_prime_C, "<=")

<IPython.core.display.Latex object>

In [64]:
if "NG" in check_compression:
    display_text("A 6x6 post will not work--we need to use HSS")